# DATA09 v0-flow 双通道TDMS特征提取

本 notebook 用于对 `E:\codes\ZZ-BK\DATA09\v0-flow` 目录下的连续数据进行特征提取。

## 数据特点
- 文件格式：TDMS双通道文件
- 数据路径：`E:\codes\ZZ-BK\DATA09\v0-flow`
- 需要读取指定通道的数据进行特征提取


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# 1 = prefer CUDA when available; 0/false/off/no = force CPU.
os.environ.setdefault('FEA_CPT_USE_GPU', '1')

workspace = Path.cwd()
if not (workspace / 'src').exists():
    workspace = workspace.parent
if str(workspace / 'src') not in sys.path:
    sys.path.insert(0, str(workspace / 'src'))

from fea_cpt_gpu_v2_2.sliding_window import (
    SlidingWindowConfig,
    build_sliding_window_dataset,
    compute_shared_stft,
    discover_source_files,
    downsample_source_files,
    gpu_backend_info,
    list_window_ranges,
    upsample_to_target,
    process_source_file,
    _auto_detect_workers,
)
from fea_cpt_gpu_v2_2.params import DEFAULT_FEATURE_PARAMS

print(f'workspace = {workspace}')
print(gpu_backend_info())
print(f'Python = {sys.version}')
print(f'CPU 核心数: {os.cpu_count()}')
print(f'推荐 workers: {_auto_detect_workers()}')
print('v2.2 所有模块加载成功')


In [ ]:
# =========================
# Global config
# =========================

# Configure each input root/file with its own sampling ratio. ratio=1.0 means all files.
# WinError 1455 on Windows is usually caused by excessive STFT batch/shared-memory pressure.
# Defaults below favor stable long runs; increase WINDOW_WORKERS/STFT_BATCH_SIZE only after a clean run.
RAW_DATA_ROOT_SPECS = [
    (r"E:\PCCP\0904-FLOW-v0.5", 1.0),
    # (r"E:\PCCP\0829-flow-v0", 1.0),
    # (r"E:\PCCP\0904-flow-v0", 0.4),
    # (r"E:\PCCP\20260902-flow-v0-a", 0.1),
    # (r"E:\PCCP\20260902-flow-v0-b", 0.01),
]


def parse_input_specs(
    specs: list[tuple[str | Path, float]] | tuple[tuple[str | Path, float], ...],
) -> list[tuple[Path, float]]:
    parsed: list[tuple[Path, float]] = []
    seen: set[str] = set()
    for raw_path, ratio in specs:
        path = Path(raw_path).expanduser()
        key = str(path.resolve()) if path.exists() else str(path)
        if key in seen:
            print(f'[WARN] duplicate input skipped: {path}')
            continue
        ratio = float(ratio)
        if not 0.0 < ratio <= 1.0:
            raise ValueError(f'sampling ratio must be in (0, 1]: {raw_path} -> {ratio}')
        parsed.append((path, ratio))
        seen.add(key)
    if not parsed:
        raise ValueError('RAW_DATA_ROOT_SPECS is empty')
    return parsed

RAW_DATA_SPECS = parse_input_specs(RAW_DATA_ROOT_SPECS)
RAW_DATA_ROOTS = [path for path, _ratio in RAW_DATA_SPECS]

WINDOW_DURATION_S = 0.03
WINDOW_OVERLAP = 0.0
assert 0.0 <= WINDOW_OVERLAP < 1.0

TARGET_SAMPLE_RATE = 1_000_000.0
PREPROC_BAND = (1_000.0, 95_000.0)

BANDS = [
    ('b_1k_100k',  (1_000.0,  100_000.0)),
    ('b_1k_10k',   (1_000.0,  10_000.0)),
    ('b_10k_20k',  (10_000.0, 20_000.0)),
    ('b_20k_30k',  (20_000.0, 30_000.0)),
    ('b_30k_40k',  (30_000.0, 40_000.0)),
    ('b_40k_60k',  (40_000.0, 60_000.0)),
    ('b_10k_50k',  (10_000.0, 50_000.0)),
    ('b_1k_50k',   (1_000.0,  50_000.0)),
]

# Conservative Windows defaults: 4 workers and 50 windows per STFT batch reduce mmap/page-file peaks.
# If WinError 1455 still appears, set STFT_BATCH_SIZE=20, WINDOW_WORKERS=2, or ENABLE_SHARED_STFT=False.
WINDOW_WORKERS = min(4, _auto_detect_workers())
WINDOW_BATCH_SIZE = 256
ENABLE_NUMA_BINDING = False
ENABLE_SHARED_STFT = True
STFT_BATCH_SIZE = 50

ENABLE_FILE_SAMPLING = True
FILE_SAMPLE_SEED = 42

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_ROOT = workspace / 'outputs' / 'DATA09_v0-flow_features'
RUN_OUTPUT_ROOT = OUTPUT_ROOT / f'run_{RUN_TIMESTAMP}'
PROCESS_LOG_DIR = OUTPUT_ROOT / '_process_logs'
PROCESSED_LIST_PATH = PROCESS_LOG_DIR / 'processed_source_files.txt'
NPZ_PER_CSV = 20
MAX_FILES: int | None = None
TDMS_FALLBACK_SAMPLE_RATE_HZ: float | None = 1_000_000.0
TARGET_CHANNEL_NAME = 'Untitled'

RUN_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PROCESS_LOG_DIR.mkdir(parents=True, exist_ok=True)

auto_workers = _auto_detect_workers() if WINDOW_WORKERS is None else WINDOW_WORKERS
print(f'input roots: {len(RAW_DATA_SPECS)}')
for root, ratio in RAW_DATA_SPECS:
    exists = 'exists' if root.exists() else 'missing'
    print(f'  - {root} (ratio={ratio:.3g}, {exists})')
print(f'window: {WINDOW_DURATION_S*1000:.0f} ms, overlap {WINDOW_OVERLAP*100:.0f}%')
print(f'target sample rate: {TARGET_SAMPLE_RATE/1000:.0f} kHz')
print(f'band count: {len(BANDS)}')
print(f'parallel: {auto_workers} workers, STFT batch={STFT_BATCH_SIZE}')
print(f'NUMA binding: {ENABLE_NUMA_BINDING}')
print(f'shared STFT: {ENABLE_SHARED_STFT}')
print(f'file sampling: {ENABLE_FILE_SAMPLING} (seed={FILE_SAMPLE_SEED})')
print(f'output dir: {RUN_OUTPUT_ROOT}')
print(f'processed log: {PROCESSED_LIST_PATH}')
print(f'TDMS channel: {TARGET_CHANNEL_NAME}')

config = SlidingWindowConfig(
    bands=BANDS,
    preproc_band=PREPROC_BAND,
    window_duration_s=WINDOW_DURATION_S,
    window_overlap=WINDOW_OVERLAP,
    target_sample_rate=TARGET_SAMPLE_RATE,
    tdms_fallback_sample_rate=TDMS_FALLBACK_SAMPLE_RATE_HZ,
    tdms_channel_name=TARGET_CHANNEL_NAME,
    window_workers=WINDOW_WORKERS,
    window_batch_size=WINDOW_BATCH_SIZE,
    enable_numa_binding=ENABLE_NUMA_BINDING,
    enable_shared_stft=ENABLE_SHARED_STFT,
    stft_batch_size=STFT_BATCH_SIZE,
)
print('\nconfig created')


In [ ]:
# =========================
# 数据文件发现与按路径抽样
# =========================

import random
from collections import Counter

def sample_files_for_root(files: list[Path], ratio: float, seed: int, root_index: int) -> list[Path]:
    if ratio >= 1.0:
        return list(files)
    if not files:
        return []
    sample_count = max(1, int(round(len(files) * ratio)))
    sample_count = min(sample_count, len(files))
    rng = random.Random(seed + root_index)
    return sorted(rng.sample(list(files), sample_count))

source_files_all: list[Path] = []
source_files_sampled: list[Path] = []
seen_files: set[str] = set()

print('按输入路径发现文件并抽样:')
for root_index, (root, ratio) in enumerate(RAW_DATA_SPECS):
    root_files = discover_source_files([root], max_files=MAX_FILES)
    sampled_files = sample_files_for_root(root_files, ratio if ENABLE_FILE_SAMPLING else 1.0, FILE_SAMPLE_SEED, root_index)

    print(
        f'  [{root_index + 1}] {root}: '
        f'发现 {len(root_files)} 个, 抽取 {len(sampled_files)} 个, 比例={ratio:.3g}'
    )

    source_files_all.extend(root_files)
    for file_path in sampled_files:
        file_key = str(file_path)
        if file_key in seen_files:
            continue
        source_files_sampled.append(file_path)
        seen_files.add(file_key)

source_files_all = sorted(dict.fromkeys(source_files_all))
source_files = sorted(source_files_sampled)

if not source_files_all:
    raise FileNotFoundError(f'未找到任何 .npz/.tdms 文件，请检查路径: {RAW_DATA_ROOTS}')

if not source_files:
    raise FileNotFoundError('抽样后没有任何待处理文件，请检查各路径抽取比例和输入路径')

if PROCESSED_LIST_PATH.exists():
    processed_set = {
        line.strip()
        for line in PROCESSED_LIST_PATH.read_text(encoding='utf-8').splitlines()
        if line.strip()
    }
else:
    processed_set = set()

source_files_before_resume = list(source_files)
source_files = [f for f in source_files if str(f) not in processed_set]

folder_counts_all = Counter(f.parent.name for f in source_files_all)
folder_counts_sampled = Counter(f.parent.name for f in source_files_before_resume)
folder_counts_pending = Counter(f.parent.name for f in source_files)

print(f'\n总发现文件: {len(source_files_all)}')
print(f'抽样后文件: {len(source_files_before_resume)}')
print(f'断点日志中已有 {len(processed_set)} 个已处理文件')
print(f'本次待处理 {len(source_files)} 个源文件')

print(f'\n各文件夹文件数:')
for folder, total_count in sorted(folder_counts_all.items()):
    sampled_count = folder_counts_sampled.get(folder, 0)
    pending_count = folder_counts_pending.get(folder, 0)
    print(f'  {folder}: {pending_count} 待处理 / {sampled_count} 抽样 / {total_count} 总数')

npz_count = sum(1 for f in source_files_all if f.suffix.lower() == '.npz')
tdms_count = sum(1 for f in source_files_all if f.suffix.lower() == '.tdms')
print(f'\n文件格式: {npz_count} npz, {tdms_count} tdms')


In [ ]:
# =========================
# Inspect TDMS structure
# =========================

if source_files:
    test_file = source_files[0]
    print(f'inspect file: {test_file.name}')

    if test_file.suffix.lower() == '.tdms':
        from nptdms import TdmsFile

        td = TdmsFile.read(test_file)
        print('\nGroups:')
        for g in td.groups():
            print(f'  {g.name}')
            for c in g.channels():
                print(f'    Channel: {c.name}')
                print(f'    Length: {len(c[:])}')
                print(f'    Properties: {dict(c.properties)}')
                print()
    else:
        print(f'first file is not TDMS: {test_file.suffix}')
else:
    print('no pending files; skip structure inspection')


In [ ]:
# =========================
# Single-file smoke test
# =========================

import time

if source_files:
    test_file = source_files[0]
    print(f'test file: {test_file.name}')

    from fea_cpt_gpu_v2_2.sliding_window import load_source_file, upsample_to_target

    src = load_source_file(test_file, config.tdms_fallback_sample_rate, config.tdms_channel_name)
    raw = np.asarray(src['signal_values'], dtype=float)
    orig_rate = float(src['sample_rate'])
    print(f'  original sample rate: {orig_rate/1000:.0f} kHz, samples: {len(raw):,}')
    print(f'  channel: {src["source_channel_name"]}')

    sig_up, eff_rate = upsample_to_target(raw, orig_rate, config.target_sample_rate)
    print(f'  after resampling: {eff_rate/1000:.0f} kHz, samples: {len(sig_up):,}')

    windows = list_window_ranges(len(sig_up), eff_rate, config.window_duration_s, config.window_overlap)
    print(f'  windows: {len(windows)}')

    print('\nstart single-file feature extraction with per-chunk STFT cleanup...')
    t0 = time.time()
    df_feat, df_log = process_source_file(test_file, config)
    elapsed = time.time() - t0

    print(f'  elapsed: {elapsed:.1f} s')
    print(f'  feature shape: {df_feat.shape}')
    missing_count = df_log['missing_selected_features'].fillna('').astype(str).ne('').sum() if 'missing_selected_features' in df_log else 0
    print(f'  windows with feature warnings: {missing_count}')
    print('\nsingle-file smoke test passed')
else:
    print('no pending files; skip single-file smoke test')


In [ ]:
# =========================
# Batch processing
# =========================

import time
from datetime import datetime

processed_list_path = PROCESSED_LIST_PATH

print(f'{"="*60}')
print('Batch processing start (v2.2 pipeline)')
print(f'Time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'File count: {len(source_files)}')
print('Progress bar covers pending files; ETA is estimated from completed files.')
print(f'Output dir: {RUN_OUTPUT_ROOT}')
print(f'Processed log: {processed_list_path}')
print(f'{"="*60}')

def _timestamped_name(csv_path: Path, prefix: str) -> Path:
    if csv_path.name.startswith(f'{prefix}_{RUN_TIMESTAMP}_'):
        return csv_path
    return csv_path.with_name(f'{prefix}_{RUN_TIMESTAMP}_{csv_path.name.removeprefix(prefix + "_")}')

t_start = time.time()

try:
    stats = build_sliding_window_dataset(
        source_paths=source_files,
        config=config,
        output_dir=RUN_OUTPUT_ROOT,
        processed_list_path=processed_list_path,
        npz_per_csv=NPZ_PER_CSV,
        show_progress=True,
    )
except OSError as exc:
    msg = str(exc)
    if getattr(exc, 'winerror', None) == 1455 or 'WinError 1455' in msg or '\u9875\u9762\u6587\u4ef6\u592a\u5c0f' in msg:
        raise RuntimeError(
            'Windows page file is too small. Reduce STFT_BATCH_SIZE to 20, reduce WINDOW_WORKERS to 2, '
            'or set ENABLE_SHARED_STFT=False, then rerun from the processed log.'
        ) from exc
    raise

t_elapsed = time.time() - t_start

for csv_path in sorted(RUN_OUTPUT_ROOT.glob('features_part_*.csv')):
    target = _timestamped_name(csv_path, 'features')
    if target != csv_path:
        csv_path.rename(target)
for csv_path in sorted(RUN_OUTPUT_ROOT.glob('log_part_*.csv')):
    target = _timestamped_name(csv_path, 'log')
    if target != csv_path:
        csv_path.rename(target)

print()
print(f'{"="*60}')
print('Batch processing complete')
print(f'Elapsed: {t_elapsed:.1f} s ({t_elapsed/60:.1f} min)')
print(f'Processed files: {stats["processed"]}')
print(f'Skipped files: {stats["skipped"]}')
print(f'Failed files/windows: {stats["failed"]}')
print(f'Total windows: {stats["windows"]}')
if stats['processed'] > 0:
    avg = t_elapsed / stats['processed']
    print(f'Average per file: {avg:.1f} s')
print(f'processed log kept at: {processed_list_path}')
print(f'{"="*60}')


In [ ]:
# =========================
# Result summary
# =========================

feature_chunks = sorted(RUN_OUTPUT_ROOT.glob(f'features_{RUN_TIMESTAMP}_part_*.csv'))
log_chunks = sorted(RUN_OUTPUT_ROOT.glob(f'log_{RUN_TIMESTAMP}_part_*.csv'))

print(f'feature CSV count: {len(feature_chunks)}')
print(f'log CSV count: {len(log_chunks)}')

if feature_chunks:
    df_sample = pd.read_csv(feature_chunks[0], nrows=5)
    print(f'feature CSV columns: {len(df_sample.columns)}')

    total_windows = 0
    total_files_set = set()
    for chunk in feature_chunks:
        df = pd.read_csv(chunk, usecols=['source_file_name', 'source_file_path', 'window_id'])
        total_windows += len(df)
        total_files_set.update(df['source_file_path'].unique())
        dup_count = df.duplicated(['source_file_path', 'window_id']).sum()
        if dup_count:
            print(f'[WARN] duplicate window keys in {chunk.name}: {dup_count}')
    print(f'total windows: {total_windows:,}')
    print(f'source files in this run: {len(total_files_set)}')

if log_chunks:
    log_summary = []
    for chunk in log_chunks:
        df_log_chunk = pd.read_csv(chunk)
        missing = df_log_chunk.get('missing_selected_features', pd.Series('', index=df_log_chunk.index)).fillna('').astype(str)
        log_summary.append({
            'log_file': chunk.name,
            'rows': len(df_log_chunk),
            'source_files': df_log_chunk['source_file_path'].nunique() if 'source_file_path' in df_log_chunk else np.nan,
            'missing_feature_windows': int(missing.ne('').sum()),
        })
    print('\nlog chunk summary:')
    print(pd.DataFrame(log_summary).to_string(index=False))


In [ ]:
# =========================
# Feature quality audit
# =========================

if feature_chunks:
    df_check = pd.read_csv(feature_chunks[0])

    meta_cols = {
        'source_file_name', 'source_file_path', 'source_format',
        'source_group_name', 'source_channel_name', 'source_detail',
        'window_id', 'window_start_index', 'window_end_index',
        'window_length_samples', 'window_step_samples',
        'window_duration_s', 'window_start_offset_s',
        'sample_rate_hz', 'original_sample_rate_hz',
        'source_n_samples', 'source_duration_s',
        'starttime_raw', 'arrival_time_raw', 'sample_type',
        'window_start_datetime', 'missing_selected_features',
    }
    feature_cols = [c for c in df_check.columns if c not in meta_cols]

    print(f'feature columns: {len(feature_cols)}')

    full_nan = [c for c in feature_cols if pd.to_numeric(df_check[c], errors='coerce').isna().all()]
    if full_nan:
        print(f'WARNING: fully-NaN feature count: {len(full_nan)}')
        print(full_nan[:30])
    else:
        print('no fully-NaN features in checked feature chunk')

    nan_stats = []
    for col in feature_cols:
        series = pd.to_numeric(df_check[col], errors='coerce')
        nan_ratio = series.isna().sum() / len(series)
        if nan_ratio > 0:
            nan_stats.append({'feature': col, 'nan_ratio': nan_ratio, 'mean': series.mean()})
    print('\nfeatures with highest NaN ratios:')
    if nan_stats:
        print(pd.DataFrame(nan_stats).sort_values('nan_ratio', ascending=False).head(30).to_string(index=False))
    else:
        print('no NaN features in checked feature chunk')

if log_chunks:
    missing_counter = Counter()
    checked_rows = 0
    for chunk in log_chunks:
        df_log_chunk = pd.read_csv(chunk, usecols=['missing_selected_features'])
        checked_rows += len(df_log_chunk)
        for text in df_log_chunk['missing_selected_features'].fillna('').astype(str):
            if text:
                missing_counter[text] += 1
    print(f'\nlog rows checked: {checked_rows:,}')
    print(f'warning/error type count: {len(missing_counter)}')
    for err, count in missing_counter.most_common(20):
        print(f'{count:8d} | {err[:180]}')


In [ ]:
# =========================
# Run logs
# =========================

processed_log = PROCESSED_LIST_PATH
failed_log = RUN_OUTPUT_ROOT / 'failed_samples.log'

if processed_log.exists():
    lines = [line for line in processed_log.read_text(encoding='utf-8').splitlines() if line.strip()]
    print(f'processed-file records: {len(lines)}')
    print('\nlast 5 processed files:')
    for line in lines[-5:]:
        print(f'  {Path(line).name}')
else:
    print('no processed-file log yet')

if failed_log.exists():
    print('\nfailed sample log for this run:')
    print(failed_log.read_text(encoding='utf-8'))
else:
    print('\nno failed sample log for this run')

print('\nTroubleshooting:')
print('- WinError 1455: lower STFT_BATCH_SIZE first, then lower WINDOW_WORKERS; defaults are 50/4.')
print('- Non-empty missing_selected_features: inspect the error-type counts in cell 8.')
print('- Resume behavior: files in processed_source_files.txt are skipped; remove it only for a full rerun.')


## 架构说明

```
主进程(GPU):
  加载文件N → 批量GPU STFT(200窗口/批) → 写入共享内存 → 加载文件N+1 → ...
       ↕ SharedMemory (signal_pre + STFT结果)
  预加载线程: 后台加载下一文件，与当前文件的特征计算并行

持久Worker1-14(CPU):
  从共享内存读取信号切片 + STFT切片 → butter_filter + ridge + ISTFT + wavelet + 特征计算 → 返回结果
```

**关键保证：所有优化不改变特征计算公式，仅改变计算调度方式。**

### 双通道TDMS处理说明

本notebook针对双通道TDMS文件进行了适配：
1. 通过 `SlidingWindowConfig.tdms_channel_name` 参数指定要读取的通道名称
2. 在notebook中设置 `TARGET_CHANNEL_NAME = 'Untitled'` 指定通道
3. 如果指定通道不存在，会回退到默认的通道选择逻辑
4. 相关修改已集成到 `fea_cpt_gpu_v2_2.sliding_window` 模块中
5. 没有 CUDA 时自动使用 CPU STFT；断点日志固定保存在 `_process_logs/processed_source_files.txt`
